# Gradient Boosting

Previous experiments showed that Random Forest substantially outperformed the Logistic Regression baseline, while additional behavioral features did not improve validation performance.

The strongest model so far is the Random Forest using the original feature set with `mcc_key` treated as a categorical feature.

This experiment evaluates Gradient Boosting as an alternative tree-based approach while keeping the feature set and time-based validation strategy consistent with the strongest previous experiment.

In [1]:
from sklearn.ensemble import GradientBoostingClassifier

from finsight.database import (
    connect_to_database,
    test_database_connection,
)

from finsight.fraud_data import (
    download_train_data,
    download_validation_data,
)

from finsight.fraud_preprocessing  import (
    encode_categorical_features,
    handle_missing_values,
)

from finsight.model_evaluation import (
    evaluate_model,
)

In [2]:
engine = connect_to_database()
test_database_connection(engine)

Connected!


## Data Preparation

To ensure a fair comparison with the strongest Random Forest experiment, the same base feature set and time-based train-validation split are used.

`mcc_key` is treated as a categorical feature and categorical variables are one-hot encoded using the training data. Missing numerical values are then imputed using training-set medians.

In [3]:
X_train, y_train = download_train_data(engine)
X_val, y_val = download_validation_data(engine)

In [4]:
X_train_mcc = X_train.copy()
X_val_mcc = X_val.copy()

X_train_mcc["mcc_key"] = X_train_mcc["mcc_key"].astype(str)
X_val_mcc["mcc_key"] = X_val_mcc["mcc_key"].astype(str)

In [5]:
final_train, final_val = encode_categorical_features(X_train_mcc, X_val_mcc)

In [6]:
final_train, final_val = handle_missing_values(final_train, final_val)

## Gradient Boosting Baseline

Gradient Boosting builds trees sequentially, with each new tree attempting to improve the errors made by the existing model.

The first experiment uses the default model configuration to establish a baseline before investigating whether parameter changes improve fraud detection performance.

The model is evaluated on the unchanged time-based validation set using recall, precision and F1-score.

In [7]:
gb_baseline = GradientBoostingClassifier(random_state=42)
gb_baseline.fit(final_train, y_train)
print("Training set score: {:.4f}".format(gb_baseline.score(final_train, y_train)))
print("Validation set score: {:.4f}".format(gb_baseline.score(final_val, y_val)))

Training set score: 0.9853
Validation set score: 0.9968


In [8]:
gradient_boosting_baseline = evaluate_model(gb_baseline, final_val, y_val, "gb_baseline")
gradient_boosting_baseline

{'experiment': 'gb_baseline',
 'recall': 0.034990791896869246,
 'precision': 0.03977669225401256,
 'f1': 0.037230568256041804,
 'confusion_matrix': array([[931594,   1376],
        [  1572,     57]])}

### Learning Rate

The baseline model uses the default `learning_rate` of 0.1.

To investigate whether more gradual learning improves fraud detection performance, the learning rate is reduced to 0.01 while the remaining model parameters are kept unchanged.

This isolates the effect of reducing the contribution of each individual tree.

In [9]:
gb_lr_001 = GradientBoostingClassifier(learning_rate=0.01, random_state=42)
gb_lr_001.fit(final_train, y_train)
print("Training set score: {:.4f}".format(gb_lr_001.score(final_train, y_train)))
print("Validation set score: {:.4f}".format(gb_lr_001.score(final_val, y_val)))

Training set score: 0.9669
Validation set score: 0.9983


In [11]:
gradient_boosting_lr001 = evaluate_model(gb_lr_001, final_val, y_val, "gb_lr_001")
gradient_boosting_lr001

{'experiment': 'gb_lr_001',
 'recall': 0.004910988336402701,
 'precision': 0.42105263157894735,
 'f1': 0.009708737864077669,
 'confusion_matrix': array([[932959,     11],
        [  1621,      8]])}

### Learning Rate and Number of Estimators

Reducing the learning rate to 0.01 substantially decreased recall, suggesting that 100 trees were insufficient when each tree contributed only a small adjustment to the model.

To test whether more gradual learning can benefit from additional boosting stages, `learning_rate` is kept at 0.01 while `n_estimators` is increased from 100 to 1000.

The remaining model parameters are kept unchanged.

In [12]:
gb_lr_001_e_1000 = GradientBoostingClassifier(learning_rate=0.01, n_estimators=1000, random_state=42)
gb_lr_001_e_1000.fit(final_train, y_train)
print("Training set score: {:.4f}".format(gb_lr_001_e_1000.score(final_train, y_train)))
print("Validation set score: {:.4f}".format(gb_lr_001_e_1000.score(final_val, y_val)))

Training set score: 0.9845
Validation set score: 0.9969


In [13]:
gradient_boosting_lr_001_estimators_1000_results = evaluate_model(gb_lr_001_e_1000, final_val, y_val, "gb_lr_001_e_1000")
gradient_boosting_lr_001_estimators_1000_results

{'experiment': 'gb_lr_001_e_1000',
 'recall': 0.03314917127071823,
 'precision': 0.03865425912670007,
 'f1': 0.0356906807666887,
 'confusion_matrix': array([[931627,   1343],
        [  1575,     54]])}

### Learning Rate and Number of Estimators Results

Increasing `n_estimators` from 100 to 1000 while keeping `learning_rate=0.01` substantially recovered the recall lost in the previous experiment.

The resulting recall (0.0331), precision (0.0387), and F1-score (0.0357) are close to the Gradient Boosting baseline, but do not improve upon it.

This suggests that the lower learning rate required more boosting stages to reach comparable performance. However, the combination of a smaller learning rate and a larger number of estimators did not provide an advantage over the default configuration.

### Tree Depth

The previous experiments showed that reducing the learning rate and increasing the number of estimators did not improve performance over the Gradient Boosting baseline.

This experiment investigates whether the baseline trees are too simple to capture more complex interactions between fraud-related features.

`max_depth` is increased while `learning_rate` and `n_estimators` remain at their baseline values. This isolates the effect of increasing the complexity of individual trees.

In [14]:
gb_md6 = GradientBoostingClassifier(max_depth=6, random_state=42)
gb_md6.fit(final_train, y_train)
print("Training set score: {:.4f}".format(gb_md6.score(final_train, y_train)))
print("Validation set score: {:.4f}".format(gb_md6.score(final_val, y_val)))

Training set score: 0.9925
Validation set score: 0.9968


In [15]:
gradient_boosting_md6_results = evaluate_model(gb_md6, final_val, y_val, "gb_md6")
gradient_boosting_md6_results

{'experiment': 'gb_md6',
 'recall': 0.09944751381215469,
 'precision': 0.09574468085106383,
 'f1': 0.0975609756097561,
 'confusion_matrix': array([[931440,   1530],
        [  1467,    162]])}

### Tree Depth Results

Increasing `max_depth` from 3 to 6 substantially improved fraud detection performance.

Recall increased from approximately 3.5% to 9.9%, while precision increased from approximately 4.0% to 9.6%. As a result, the F1-score increased from 0.037 to approximately 0.098.

This suggests that the shallow trees used by the baseline Gradient Boosting model were not sufficiently expressive to capture more complex relationships between the transaction and behavioral features.

Unlike the previous learning-rate and estimator experiments, increasing tree complexity produced a clear improvement over the baseline. Therefore, tree depth appears to be an important parameter for this dataset.

In [ ]:
gb_md10 = GradientBoostingClassifier(max_depth=10, random_state=42)
gb_md10.fit(final_train, y_train)
print("Training set score: {:.4f}".format(gb_md10.score(final_train, y_train)))
print("Validation set score: {:.4f}".format(gb_md10.score(final_val, y_val)))

In [ ]:
gradient_boosting_md10_results = evaluate_model(gb_md10, final_val, y_val, "gb_md10")
gradient_boosting_md10_results